Run this cell to programmatically restart the runtime and clear existing processes:

In [ ]:
import os
import signal

# Restart the runtime programmatically
os.kill(os.getpid(), signal.SIGKILL)

In [ ]:
# 1. Install compatible vLLM and Torch versions
!pip install vllm==0.6.3 torch==2.4.0 --index-url https://download.pytorch.org/whl/cu121 -q

import os
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

import torch
import multiprocessing

try:
    multiprocessing.set_start_method('spawn', force=True)
except RuntimeError:
    pass

if not torch.cuda.is_available():
    print("No GPU detected. vLLM requires a GPU to run.")
else:
    from vllm import LLM, SamplingParams
    model_name = "BioMistral/BioMistral-7B"

    llm = LLM(
        model=model_name,
        dtype="auto",
        trust_remote_code=True,
        gpu_memory_utilization=0.90,
        enforce_eager=True
    )

    sampling_params = SamplingParams(
        temperature=0.7,
        top_p=0.95,
        max_tokens=256
    )

    prompts = ["Hastanın tıbbi geçmişini özetleyin:"]
    outputs = llm.generate(prompts, sampling_params)

    for output in outputs:
        print(f"Prompt: {output.prompt}\nGenerated text: {output.outputs[0].text}")

# Task
Evaluate the BioMistral model via vLLM on the biomedical datasets.

In [ ]:
import os
import json

base_dir = '/content/drive/MyDrive/biomedical_datasets'
dataset_mapping = {}

if os.path.exists(base_dir):
    for root, dirs, files in os.walk(base_dir):
        for file in files:
            if file.endswith(('.jsonl', '.json', '.csv')):
                file_path = os.path.join(root, file)
                category = os.path.basename(root)
                if 'qa' in file.lower() or 'support' in category.lower() or 'decision' in category.lower():
                    metrics = ['Accuracy']
                else:
                    metrics = ['ROUGE', 'BERTScore']
                dataset_mapping[file_path] = {'category': category, 'metrics': metrics}

print("Dataset Mapping completed.")